# VascuQuest Parameterized Cohort Manual Qualification

This notebook is the checkpointed real-PWDB qualification route for PR #20.

Run in order: setup/source staging → Phase A (1 subject × 4 diseases) → Phase B (3 subjects × 4 diseases) → finalize.

Every completed subject is persisted and verified. PASS remains MODELLED and is not clinical validation.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, shutil, subprocess, sys, json

REPO_URL = "https://github.com/KNOWDYN/VascuQuest.git"
QUALIFICATION_REF = "release/parameterized-cohort-qualification"

# Search all of MyDrive for checksum-valid existing PWDB files. Any required
# artifact not present (e.g. geo.zip) is acquired through VascuQuest's canonical
# verified artifact layer and staged to local SSD.
DRIVE_SEARCH_ROOT = Path("/content/drive/MyDrive")
OUTPUT_BASE = Path("/content/drive/MyDrive/VascuQuest/parameterized_cohort_qualification")
LOCAL_REPO = Path("/content/VascuQuest-qualification")
LOCAL_SOURCE = Path("/content/vascuquest-pwdb-source")
LOCAL_XDG = Path("/content/vascuquest-xdg")

OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
LOCAL_SOURCE.mkdir(parents=True, exist_ok=True)
LOCAL_XDG.mkdir(parents=True, exist_ok=True)

os.environ["XDG_DATA_HOME"] = str(LOCAL_XDG / "data")
os.environ["XDG_CACHE_HOME"] = str(LOCAL_XDG / "cache")
os.environ["XDG_STATE_HOME"] = str(LOCAL_XDG / "state")

if LOCAL_REPO.exists():
    shutil.rmtree(LOCAL_REPO)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", QUALIFICATION_REF, REPO_URL, str(LOCAL_REPO)],
    check=True,
)
CODE_REVISION = subprocess.check_output(
    ["git", "-C", str(LOCAL_REPO), "rev-parse", "HEAD"], text=True
).strip()

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(LOCAL_REPO)],
    check=True,
)

RUNNER = LOCAL_REPO / "tests/full_data/parameterized_cohort_colab_validation.py"
STAGER = LOCAL_REPO / "tests/full_data/parameterized_cohort_colab_stage.py"
OUTPUT_ROOT = OUTPUT_BASE / CODE_REVISION[:12]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Code revision:", CODE_REVISION)
print("Revision-scoped output root:", OUTPUT_ROOT)

# Prepare the complete local source before Phase A.
stage_cmd = [
    sys.executable, str(STAGER),
    "--drive-search-root", str(DRIVE_SEARCH_ROOT),
    "--local-source", str(LOCAL_SOURCE),
    "--report", str(OUTPUT_ROOT / "source_stage.json"),
]
stage = subprocess.run(stage_cmd)
if stage.returncode != 0:
    raise RuntimeError(f"PWDB source staging failed with exit code {stage.returncode}")

print((OUTPUT_ROOT / "source_stage.json").read_text())
print("PWDB local-SSD source gate: PASS")

## Phase A — rapid smoke qualification

Runs 1 subject for each of the four disease conditions. This must pass before Phase B.

In [ ]:
def run_qualification_phase(phase: str):
    result_name = "smoke_results.json" if phase == "smoke" else "full_results.json"
    cmd = [
        sys.executable, str(RUNNER),
        "--phase", phase,
        # The staging helper already created a complete checksum-verified local source.
        # Passing LOCAL_SOURCE as both roots prevents another Drive scan/download.
        "--drive-pwdb-root", str(LOCAL_SOURCE),
        "--local-source", str(LOCAL_SOURCE),
        "--output-root", str(OUTPUT_ROOT),
        "--code-revision", CODE_REVISION,
    ]
    completed = subprocess.run(cmd)
    result_path = OUTPUT_ROOT / result_name
    if completed.returncode != 0:
        if result_path.exists():
            print("\nPersisted failure record:")
            print(result_path.read_text())
        raise RuntimeError(
            f"{phase} qualification failed with exit code {completed.returncode}; "
            f"see output above and {result_path}"
        )
    payload = json.loads(result_path.read_text())
    print(json.dumps(
        {"status": payload["status"], "cases_completed": payload["cases_completed"]},
        indent=2,
    ))
    return payload

smoke = run_qualification_phase("smoke")

In [ ]:
smoke = json.loads((OUTPUT_ROOT / "smoke_results.json").read_text())
if smoke.get("status") != "PASS" or smoke.get("cases_completed") != 4:
    raise RuntimeError("Smoke qualification is not complete/PASS.")
print("Smoke gate: PASS — full qualification authorized.")

## Phase B — 3 subjects × 4 diseases

Completed subjects are checkpointed to Drive. Rerun this cell to resume.

In [ ]:
full = run_qualification_phase("full")

## Finalize machine-readable qualification evidence

In [ ]:
cmd = [
    sys.executable, str(RUNNER),
    "--phase", "finalize",
    "--drive-pwdb-root", str(LOCAL_SOURCE),
    "--local-source", str(LOCAL_SOURCE),
    "--output-root", str(OUTPUT_ROOT),
    "--code-revision", CODE_REVISION,
]
completed = subprocess.run(cmd)
if completed.returncode != 0:
    raise RuntimeError(f"finalize failed with exit code {completed.returncode}")

report_path = OUTPUT_ROOT / "parameterized-cohort-qualification.json"
report = json.loads(report_path.read_text())
print(json.dumps({
    "status": report["status"],
    "code_revision": report["code_revision"],
    "report": str(report_path),
    "scientific_boundary": report["scientific_boundary"],
}, indent=2))

After the final cell prints `PASS`, use the revision-scoped `parameterized-cohort-qualification.json` as the evidence for deciding whether PR #20 is ready to merge.